# Saliendo de lo Pandito
## Módulo 00: Debugging Asistido con IA

### 📘 Objetivos de Este Notebook:
1. Aprender a **describir errores efectivamente** a Genie Code
2. Dominar la **interpretación de stacktraces** y mensajes de error
3. Resolver **problemas de performance** con asistencia de IA
4. Depurar **lógica de negocio** incorrecta
5. Practicar con **casos reales** de debugging

---

### 🎯 ¿Por Qué Este Notebook?

**La realidad del desarrollo:**
- 70% del tiempo se gasta en debugging, no en escribir código
- Los errores crípticos paralizan el aprendizaje
- Google/Stack Overflow no siempre tienen tu error exacto

**Con Genie Code:**
- Resuelves errores 5x más rápido
- Entiendes la causa raíz, no solo el síntoma
- Aprendes patrones de prevención

## 🐛 Anatomía de un Error

### Los 3 Componentes de Todo Error

```
1. TIPO DE ERROR (What)
   ↓
2. MENSAJE DESCRIPTIVO (Why)
   ↓  
3. STACKTRACE (Where)
```

### Ejemplo Real Desglosado

```python
KeyError: 'ventas_totales'
^^^^^^^^^^^  ^^^^^^^^^^^^^^
    │              │
    │              └─ Mensaje: La clave que buscaste
    └─ Tipo: Error de clave de diccionario

Stacktrace:
  File "<command-123>", line 4, in <module>
    total = df['ventas_totales'].sum()
            ^^                    ^^^^^
            └─ Línea exacta del error
```

---

### 🔍 Tipos de Errores Comunes

| Error | Causa Común | Solución Típica |
|-------|-------------|------------------|
| `KeyError` | Columna no existe | Verificar `df.columns` |
| `TypeError` | Tipos incompatibles | Convertir tipos con `.astype()` |
| `ValueError` | Valor inválido | Validar input antes de operar |
| `IndexError` | Índice fuera de rango | Verificar longitud con `len()` |
| `AttributeError` | Método no existe | Verificar documentación o tipo correcto |
| `SyntaxError` | Código inválido | Revisar paréntesis, comillas, indentación |
| `NameError` | Variable no definida | Definir variable antes de usar |

## 💬 Cómo Describir Errores a Genie Code

### ❌ Prompt Inefectivo
```
"Tengo un error, ayúdame"
```
**Problema:** Demasiado vago, Genie no tiene contexto.

---

### ✅ Prompt Efectivo - Estructura CEAS

```
[C]ontexto: Qué estabas intentando hacer
[E]rror: Mensaje de error completo
[A]cción: Qué ya intentaste
[S]olicitud: Qué necesitas específicamente
```

### Ejemplo Completo

```
Contexto: Estoy calculando el revenue total por producto desde un DataFrame
         de ventas que tiene columnas: producto, cantidad, precio_unitario

Error: KeyError: 'revenue'
       Traceback:
       Line 5: total = df['revenue'].sum()

Acción: Verifique con df.columns y 'revenue' no aparece en la lista.
        Intenté df['Revenue'] con mayúscula pero sigue fallando.

Solicitud: ¿Cómo creo la columna 'revenue' multiplicando cantidad * precio_unitario
           y luego calculo el total por producto?
```

**Resultado:** Genie genera código exacto que necesitas + explicación educativa.

In [0]:
# 💻 EJERCICIO 1: Debugging de KeyError

import pandas as pd

# DataFrame de ejemplo con error intencional
df = pd.DataFrame({
    'producto': ['Laptop', 'Mouse', 'Teclado', 'Monitor'],
    'cantidad': [10, 50, 30, 15],
    'precio_unitario': [1200, 25, 80, 350]
})

# Código con error intencional
try:
    # ¡Este código va a fallar!
    total_revenue = df['revenue'].sum()  # ERROR: columna 'revenue' no existe
    print(f"Total Revenue: ${total_revenue:,.2f}")
except KeyError as e:
    print(f"❌ KeyError detectado: {e}")
    print(f"🔍 Columnas disponibles: {df.columns.tolist()}")
    print("\n👉 TAREA: Usa Genie Code para corregir este error")
    print("   Prompt sugerido: 'Crea la columna revenue como cantidad * precio_unitario'")

## 🔍 Debugging de Performance

### 🐢 Consultas Lentas: El Problema Más Común

#### 🚩 Síntomas
- Consulta tarda más de 30 segundos
- Notebook se congela
- "Out of Memory" errors

#### 🔧 Estrategias de Diagnóstico con Genie

**Prompt Pattern:**
```
"Esta consulta [PEGAR SQL/CÓDIGO] tarda [TIEMPO].
La tabla tiene [N] filas y [M] columnas.
¿Qué optimizaciones sugieres?"
```

---

### ⚡ Optimizaciones Comunes que Genie Sugiere

#### 1️⃣ Filtrar Antes de Agregar
```python
# ❌ Lento: Agrupa todo, luego filtra
df.groupby('region').sum()[df['region'] == 'LATAM']

# ✅ Rápido: Filtra primero, luego agrupa
df[df['region'] == 'LATAM'].groupby('region').sum()
```

#### 2️⃣ Usar Columnas Específicas (No SELECT *)
```python
# ❌ Lento: Lee todas las columnas
df = spark.read.table("ventas").select("*")

# ✅ Rápido: Solo columnas necesarias
df = spark.read.table("ventas").select("producto", "revenue")
```

#### 3️⃣ Particionar por Fecha
```python
# ❌ Lento: Escanea toda la tabla
df = spark.read.table("logs").filter("fecha >= '2024-01-01'")

# ✅ Rápido: Usa particiones
df = spark.read.table("logs").filter("year=2024 AND month=1")
```

In [0]:
# 💻 EJERCICIO 2: Optimización de Consulta Lenta

import pandas as pd
import time

# Crear dataset más grande para simular lentitud
df_grande = pd.DataFrame({
    'fecha': pd.date_range('2023-01-01', periods=100000, freq='1H'),
    'region': ['Norte', 'Sur', 'Este', 'Oeste'] * 25000,
    'producto': ['A', 'B', 'C', 'D', 'E'] * 20000,
    'revenue': range(100000)
})

print(f"Dataset: {len(df_grande):,} filas")

# ❌ Código ineficiente
start = time.time()
resultado_lento = df_grande.groupby(['region', 'producto']).agg({
    'revenue': ['sum', 'mean', 'count']
}).reset_index()
tiempo_lento = time.time() - start

print(f"\n❌ Tiempo sin optimizar: {tiempo_lento:.4f} segundos")
print(f"Resultado shape: {resultado_lento.shape}")

# 👉 TAREA: Usa Genie para optimizar esta consulta
print("\n👉 PROMPT SUGERIDO:")
print("   'Este código agrupa 100K filas. ¿Cómo lo optimizo?'")
print("   'Sugerencias: filtrar primero, usar columnas específicas, etc.'")

## 🧠 Debugging de Lógica de Negocio

### 🔴 El Error Más Peligroso: Código que "Funciona" Pero Está Mal

```python
# ¿Este código calcula el revenue correcto?
revenue = df['cantidad'] + df['precio_unitario']  # 🚨 SUMA en vez de MULTIPLICAR
```

**Problema:** No hay error de Python, pero la lógica está mal.

---

### 🔍 Cómo Usar Genie para Validar Lógica

#### Prompt Pattern de Revisión
```
"Revisa este código que calcula [DESCRIPCIÓN DE NEGOCIO]:

[PEGAR CÓDIGO]

¿La lógica es correcta? ¿Hay edge cases que no considero?"
```

### Ejemplo Real

**Prompt:**
```
"Revisa este código que calcula el Churn Rate mensual:

churn_rate = usuarios_cancelados / usuarios_totales * 100

¿La lógica es correcta?"
```

**Respuesta de Genie:**
```
🚨 Problema detectado:

La fórmula es correcta, pero falta considerar:
1. ¿usuarios_totales incluye nuevos usuarios del mes?
2. Debería ser: cancelados / activos_al_inicio_del_mes

Código corregido:

activos_inicio = df[df['status'] == 'activo'].groupby('mes').first()
cancelados = df[df['status'] == 'cancelado'].groupby('mes').count()
churn_rate = (cancelados / activos_inicio * 100).fillna(0)
```

## 📁 Casos de Estudio: Errores Reales y Sus Soluciones

### Caso 1: "SettingWithCopyWarning" en Pandas

**Error:**
```python
SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame
```

**Prompt a Genie:**
```
"Tengo este warning en Pandas:

df_filtrado = df[df['edad'] > 18]
df_filtrado['categoria'] = 'adulto'

SettingWithCopyWarning...

¿Cómo lo corrijo?"
```

**Solución de Genie:**
```python
# Usa .copy() explícitamente
df_filtrado = df[df['edad'] > 18].copy()
df_filtrado['categoria'] = 'adulto'

# O usa .loc[] para modificar in-place
df.loc[df['edad'] > 18, 'categoria'] = 'adulto'
```

---

### Caso 2: "AnalysisException" en PySpark

**Error:**
```
AnalysisException: Column 'ventas_Q1' does not exist.
Did you mean one of the following? [ventas_q1, ventas_Q2]
```

**Prompt a Genie:**
```
"PySpark me dice que la columna 'ventas_Q1' no existe,
pero la veo en df.printSchema(). ¿Por qué?"
```

**Respuesta de Genie:**
```
PySpark es case-sensitive en columnas.
'ventas_Q1' ≠ 'ventas_q1'

Solución:
1. Usa el nombre exacto: df.select('ventas_q1')
2. O renombra: df = df.withColumnRenamed('ventas_q1', 'ventas_Q1')
```

---

### Caso 3: "Out of Memory" en Joins

**Error:**
```
java.lang.OutOfMemoryError: GC overhead limit exceeded
```

**Prompt a Genie:**
```
"Este join me da OutOfMemory:

df_grande = spark.read.table('ventas')  # 100M rows
df_pequena = spark.read.table('productos')  # 1K rows
resultado = df_grande.join(df_pequena, 'producto_id')

¿Cómo lo optimizo?"
```

**Solución de Genie:**
```python
from pyspark.sql.functions import broadcast

# Usa broadcast join para tabla pequeña
resultado = df_grande.join(
    broadcast(df_pequena), 
    'producto_id'
)
```

## 🎯 Ejercicio Integrador: Debugging Challenge

### 🔴 Escenario: Reporte de Ventas Mensual Roto

Tu equipo reporta que el dashboard de ventas muestra números incorrectos.
El código está abajo. **Tiene 3 errores diferentes.**

### 📝 Tu Misión

1. **Identifica los 3 errores** (sin ejecutar el código)
2. **Usa Genie** para verificar tu hipótesis
3. **Corrige el código** con la ayuda de Genie
4. **Valida** que los resultados ahora sean correctos

### 👉 Siguiente Celda: Código con Errores

**Pistas:**
- Un error de lógica de negocio
- Un error de tipo de datos
- Un error de nombres de columnas

In [0]:
# 🐛 CÓDIGO CON 3 ERRORES - ENCUÉNTRALOS CON GENIE

import pandas as pd

df_ventas = pd.DataFrame({
    'fecha': ['2024-01-15', '2024-01-20', '2024-02-10', '2024-02-25'],
    'producto': ['Laptop', 'Mouse', 'Teclado', 'Monitor'],
    'cantidad': [10, 50, 30, 15],
    'precio_unit': [1200, 25, 80, 350],
    'descuento_pct': [10, 5, 0, 15]  # Porcentaje de descuento
})

print("Dataset original:")
print(df_ventas)
print("\n" + "="*60)

# ERROR 1: Lógica incorrecta de cálculo de revenue
# El revenue debería ser: cantidad * precio_unit * (1 - descuento_pct/100)
# Pero el código hace:
df_ventas['revenue'] = df_ventas['cantidad'] + df_ventas['precio_unit']  # 🚨

# ERROR 2: Columna mal escrita (case-sensitive)
try:
    df_ventas['mes'] = pd.to_datetime(df_ventas['Fecha']).dt.month  # 🚨
except KeyError as e:
    print(f"\n❌ ERROR 2 detectado: {e}")
    # TAREA: Corrígelo con Genie

# ERROR 3: Tipo de datos incorrecto para operación numérica
try:
    revenue_total = df_ventas['revenue'].sum()
    descuento_promedio = df_ventas['descuento_pct'].mean()  # Esto funciona
    
    # Intentar calcular revenue con descuento (fallará por ERROR 1)
    print(f"\nRevenue Total (INCORRECTO): ${revenue_total:,.2f}")
    print(f"Descuento Promedio: {descuento_promedio}%")
except Exception as e:
    print(f"\n❌ ERROR 3: {type(e).__name__}: {e}")

print("\n" + "="*60)
print("👉 TAREA: Usa Genie Code para corregir los 3 errores")
print("\nPrompt sugerido:")
print("\"Revisa este código de cálculo de revenue. Hay 3 errores:")
print("1. Lógica de cálculo incorrecta")
print("2. Nombre de columna mal escrito")
print("3. [Detectar al ejecutar]")
print("\nCorrige el código completo.\"")

## 🎓 Conclusiones y Próximos Pasos

### ✅ Lo Que Aprendiste en Este Notebook

1. **Estructura CEAS** para describir errores efectivamente
2. **Interpretación de stacktraces** y mensajes de error
3. **Debugging de performance** con patrones de optimización
4. **Validación de lógica de negocio** con IA
5. **Casos reales** de errores comunes y sus soluciones

---

### 💪 Checklist de Dominio

¿Puedes hacer esto sin dudar?

* ☑️ Describir un error a Genie usando CEAS
* ☑️ Leer un stacktrace e identificar la línea problemática
* ☑️ Diagnosticar consultas lentas con Genie
* ☑️ Pedir a Genie que valide lógica de negocio
* ☑️ Resolver 3 errores distintos en < 5 minutos con IA

Si marcaste todo, 🎉 **¡Dominas debugging asistido por IA!**

---

### 🚀 Próximo Notebook

**➡️ [00_04_Generacion_Codigo_PySpark_SQL](#notebook-00_04)**

Aprende a:
* Generar código PySpark desde cero con Genie
* Migrar de Pandas a PySpark automáticamente
* Optimizar consultas SQL complejas
* Dominar patrones de ETL con asistencia de IA

---

### 📚 Recursos Adicionales

* **Anexo de Troubleshooting:** [/anexos/troubleshooting/COMMON_ERRORS.md](#file-COMMON_ERRORS.md)
* **PySpark Errors:** [Documentación oficial](https://spark.apache.org/docs/latest/sql-error-conditions.html)
* **Pandas Warnings:** [Guía de mejores prácticas](https://pandas.pydata.org/docs/user_guide/indexing.html#returning-a-view-versus-a-copy)

---

<div style="background: linear-gradient(90deg, #f093fb 0%, #f5576c 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🔧 Debugging Ya No Es Frustrante</h3>
  <p><i>"Con Genie Code, cada error es una oportunidad de aprendizaje instantánea."</i></p>
</div>